In [1]:
import os
import duckdb
import pandas as pd

In [2]:
con = duckdb.connect()
ruta_silver = "../Data/silver/weather_clean.parquet"

# Query para procesar la capa Gold de alertas
query_gold_alertas = f"""
    SELECT 
        fecha_hora,
        temperatura_celsius,
        lluvia_mm,
        humedad_porcentaje,
        CASE 
            WHEN temperatura_celsius >= 38.0 THEN '🔴 ALERTA: Calor Extremo'
            WHEN temperatura_celsius <= 8.0 THEN '🔵 ALERTA: Frío Intenso'
            WHEN lluvia_mm >= 15.0 THEN '🌧️ ALERTA: Lluvia Fuerte / Tormenta'
            ELSE '🟢 Sin Alertas'
        END AS estado_alerta,
        CASE 
            WHEN temperatura_celsius >= 38.0 OR temperatura_celsius <= 8.0 OR lluvia_mm >= 15.0 THEN 1
            ELSE 0
        END AS tiene_alerta
    FROM read_parquet('{ruta_silver}')
    ORDER BY fecha_hora ASC
"""

df_gold_alertas = con.execute(query_gold_alertas).df()

# Ver si encontramos alguna alerta en el pronóstico actual
alertas_detectadas = df_gold_alertas[df_gold_alertas['tiene_alerta'] == 1]
print(f"Se detectaron {len(alertas_detectadas)} horas con alertas climatológicas.")
if not alertas_detectadas.empty:
    print(alertas_detectadas[['fecha_hora', 'temperatura_celsius', 'lluvia_mm', 'estado_alerta']].head())

Se detectaron 7 horas con alertas climatológicas.
            fecha_hora  temperatura_celsius  lluvia_mm           estado_alerta
6  2026-06-25 06:00:00                  7.6        0.0  🔵 ALERTA: Frío Intenso
7  2026-06-25 07:00:00                  7.2        0.0  🔵 ALERTA: Frío Intenso
8  2026-06-25 08:00:00                  7.1        0.0  🔵 ALERTA: Frío Intenso
9  2026-06-25 09:00:00                  8.0        0.0  🔵 ALERTA: Frío Intenso
30 2026-06-26 06:00:00                  8.0        0.0  🔵 ALERTA: Frío Intenso


In [3]:
query_gold_resumen = f"""
    SELECT 
        CAST(fecha_hora AS DATE) AS fecha,
        ROUND(MAX(temperatura_celsius), 1) AS temp_maxima,
        ROUND(MIN(temperatura_celsius), 1) AS temp_minima,
        ROUND(AVG(humedad_porcentaje), 1) AS humedad_promedio,
        ROUND(SUM(lluvia_mm), 1) AS lluvia_total_mm,
        MAX(estado_alerta) AS alerta_del_dia -- Toma la alerta más crítica si existe
    FROM ({query_gold_alertas})
    GROUP BY CAST(fecha_hora AS DATE)
    ORDER BY fecha ASC
"""

df_gold_resumen = con.execute(query_gold_resumen).df()
print("\nResumen diario para el Dashboard (Capa Gold):")
print(df_gold_resumen.head(7))


Resumen diario para el Dashboard (Capa Gold):
       fecha  temp_maxima  temp_minima  humedad_promedio  lluvia_total_mm  \
0 2026-06-25    16.500000          7.1              77.3              0.0   
1 2026-06-26    18.200001          7.7              77.3              0.0   
2 2026-06-27    22.100000         10.6              84.1              0.0   
3 2026-06-28    18.400000         13.3              95.1              5.9   
4 2026-06-29    17.500000          9.9              79.5              0.0   
5 2026-06-30    15.900000         10.5              87.9              3.1   
6 2026-07-01    17.000000         13.5              97.3              7.7   

  alerta_del_dia  
0  🟢 Sin Alertas  
1  🟢 Sin Alertas  
2  🟢 Sin Alertas  
3  🟢 Sin Alertas  
4  🟢 Sin Alertas  
5  🟢 Sin Alertas  
6  🟢 Sin Alertas  


In [4]:
ruta_gold_alertas = "../Data/gold/weather_alerts.parquet"
ruta_gold_resumen = "../Data/gold/weather_daily_summary.parquet"

os.makedirs("../Data/gold/", exist_ok=True)

# Limpieza por si ya existían de corridas previas
for ruta in [ruta_gold_alertas, ruta_gold_resumen]:
    if os.path.exists(ruta):
        os.remove(ruta)

# Guardar usando DuckDB
con.execute(f"COPY (SELECT * FROM df_gold_alertas) TO '{ruta_gold_alertas}' (FORMAT PARQUET);")
con.execute(f"COPY (SELECT * FROM df_gold_resumen) TO '{ruta_gold_resumen}' (FORMAT PARQUET);")

print("¡Archivos de la capa GOLD generados con éxito!")

¡Archivos de la capa GOLD generados con éxito!
